# 7-3 Transform and Preprocessing Flow — Advanced Practice

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
# 평균과 표준편차는 train Tensor에서 한 번만 계산하고 transform 객체의 상태로 고정합니다.
import torch

train = torch.tensor([1.0, 2.0, 3.0])
valid = torch.tensor([9.0, 10.0, 11.0])
train_mean = train.mean()
train_std = train.std(unbiased=False)

leaky_valid = (valid - valid.mean()) / valid.std(unbiased=False)
contract_valid = (valid - train_mean) / train_std

# 두 방식으로 변환한 validation 평균을 비교해 split 자체 통계가 분포 이동을 숨기는지 확인합니다.
print(f"own_stats_valid_mean={leaky_valid.mean().item():.1f}")
print(f"train_stats_valid_mean={contract_valid.mean().item():.4f}")
print("approved=train_stats")

own_stats_valid_mean=0.0
train_stats_valid_mean=9.7980
approved=train_stats


In [3]:
import torch
from torch.utils.data import Dataset, TensorDataset, random_split

class SubsetWithTransform(Dataset):
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base_dataset[self.indices[idx]]
        if self.transform is not None:
            x = self.transform(x)
        return x, y

# 실제 이미지 프로젝트에서는 transform=None인 원본 Dataset을 준비합니다.
X = torch.arange(100 * 6, dtype=torch.float32).reshape(100, 6)
y = torch.arange(100) % 3
raw_dataset = TensorDataset(X, y)

split_generator = torch.Generator().manual_seed(42)
train_part, valid_part, test_part = random_split(
    raw_dataset,
    [70, 15, 15],
    generator=split_generator,
)

# 예시용 transform입니다. 실제 이미지라면 train_transform과 valid_transform을 넣습니다.
train_transform = lambda x: x + torch.rand_like(x) * 0.01
valid_transform = lambda x: x
test_transform = lambda x: x

train_dataset = SubsetWithTransform(raw_dataset, train_part.indices, train_transform)
valid_dataset = SubsetWithTransform(raw_dataset, valid_part.indices, valid_transform)
test_dataset = SubsetWithTransform(raw_dataset, test_part.indices, test_transform)

print(len(train_dataset), len(valid_dataset), len(test_dataset))
print("valid sample is stable:", torch.equal(valid_dataset[0][0], valid_dataset[0][0]))

70 15 15
valid sample is stable: True


In [4]:
# 검증 가능 정답 코드
# 두 후보는 같은 train/valid 원본과 같은 통계를 사용하고 transform 순서만 바꿔 원인을 분리합니다.
import torch

x = torch.tensor([-5.0, 5.0, 15.0])
mean, std = 5.0, 5.0

# 원시 단위의 계약을 먼저 적용한 뒤 모델 입력 공간으로 바꿉니다.
candidate_a = (x.clamp(0.0, 10.0) - mean) / std
# B는 표준화된 값에 원시 범위를 적용해 음수 정보를 잘못 제거합니다.
candidate_b = ((x - mean) / std).clamp(0.0, 10.0)

# 원시 단위 clamp 계약을 지킨 후보와 위반 사유를 함께 남겨 결과 숫자만으로 승인하지 않습니다.
print(f"candidate_A={candidate_a.tolist()}")
print(f"candidate_B={candidate_b.tolist()}")
print("approved=A")

candidate_A=[-1.0, 0.0, 1.0]
candidate_B=[0.0, 0.0, 2.0]
approved=A
